# 5.3 Lightweight baselines

* R3.8 (isolation forest vs benign-Mahalanobis), R2.4 (lightweight)
* Aynı source-rate temsili, LOACO, source-disjoint, benign'de eğitim.

## FinalBaselines altyapısı

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from numpy.linalg import pinv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)
DATA="/data/"
SEEDS=(42,1,2,3,4); RS=42; FPR_GRID=(0.05,0.10,0.20)
UAV_CFG={"path":DATA+"UAVIDS-2025.csv","label_col":"label","normal":"Normal Traffic",
  "attacks":["Blackhole Attack","Flooding Attack","Sybil Attack","Wormhole Attack"],
  "leak_clean":["FlowID","SrcAddr","DstAddr","Protocol"]}
ID_CFG={"src":"SrcAddr","dst":"DstAddr"}; TARGET="Sybil Attack"
RATE=["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]
print("config ok")

config ok


In [ ]:
def load(cfg): return pd.read_csv(cfg["path"],low_memory=False).reset_index(drop=True)
def source_rate_features(df,idcfg):
    src=idcfg["src"]; dst=idcfg["dst"]
    s=df[src].astype(str).fillna("NA").values; dd=df[dst].astype(str).fillna("NA").values
    tmp=pd.DataFrame({"s":s,"d":dd}); g=tmp.groupby("s")
    flowcount=g["s"].transform("size").astype(float).values; fanout=g["d"].transform("nunique").astype(float).values
    ent_map={}
    for k,gg in tmp.groupby("s"):
        vc=gg["d"].value_counts().values.astype(float); p=vc/vc.sum(); ent_map[k]=float(-(p*np.log(p+1e-12)).sum())
    ent=np.array([ent_map[x] for x in s]); fc=np.clip(flowcount,1,None); fo=np.clip(fanout,1,None)
    return pd.DataFrame({"s_fanout_rate":fanout/fc,"s_flows_per_dst":flowcount/fo,
                         "s_dst_entropy_norm":ent/np.log(np.clip(fanout,2,None))}).fillna(0.0).reset_index(drop=True)
def feats_perflow(df,cfg):
    # CLEAN per-flow: drop leak_clean, label, AND any source-rate columns
    drop=set(cfg["leak_clean"])|{cfg["label_col"]}|set(RATE)
    X=df.drop(columns=[c for c in df.columns if c in drop],errors="ignore").copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c]=LabelEncoder().fit_transform(X[c].astype(str))
    return X.astype(float).reset_index(drop=True)
def loaco_split(df,label_col,target,seed,val=0.3,test=0.3):
    rng=np.random.default_rng(seed)
    dft=df[df[label_col].astype(str)==target]; dfk=df[df[label_col].astype(str)!=target]
    idx=rng.permutation(len(dfk)); nv=int(len(idx)*val)
    valk=dfk.iloc[idx[:nv]]; tr=dfk.iloc[idx[nv:]]; nt=int(len(tr)*test)
    return tr.iloc[nt:], valk, pd.concat([tr.iloc[:nt],dft])
def ae_(n,seed):
    b=max(2,n//4)
    return MLPRegressor(hidden_layer_sizes=(max(8,n//2),b,max(8,n//2)),max_iter=150,early_stopping=True,n_iter_no_change=8,random_state=seed)
def ae_err(ae,X):
    r=ae.predict(X); r=r.reshape(-1,1) if r.ndim==1 else r; return np.mean((X-r)**2,1)
def fit_mahal(Xn):
    mu=Xn.mean(0); cov=np.cov(Xn.T)+1e-6*np.eye(Xn.shape[1]); return mu,pinv(cov)
def mahal(X,mu,P):
    d=X-mu; return np.einsum('ij,jk,ik->i',d,P,d)
def Z(A,Bv,Be):
    imp=SimpleImputer(strategy="mean").fit(A); sca=StandardScaler().fit(imp.transform(A))
    return sca.transform(imp.transform(A)),sca.transform(imp.transform(Bv)),sca.transform(imp.transform(Be))
print("helpers ok")

helpers ok


In [ ]:
def _Z(A, Be):               # impute+scale on A (train), apply to Be (test)
    imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)), sc.transform(imp.transform(Be))

## Run Lightweight Baselines

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score

RATE = ["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]

def run_lightweight(seeds=SEEDS):
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=ID_CFG["src"]
    df=load(cfg); df=pd.concat([df, source_rate_features(df,ID_CFG)],axis=1)
    out={"benign-Mahalanobis":[], "Isolation Forest":[], "One-Class SVM":[]}
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); bmask=(ytr==normal)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        yb=is_t[sdj].astype(int)
        A=tr[RATE].reset_index(drop=True); B=test_[RATE].reset_index(drop=True)
        Za,Ze=_Z(A,B)
        Xb=Za[bmask]                       # sadece benign'de eğit (hepsi normal-anchored)

        # 1) benign-Mahalanobis (referans, mevcut yöntemimiz)
        mu,P=fit_mahal(Xb); s=mahal(Ze,mu,P)
        out["benign-Mahalanobis"].append(roc_auc_score(yb, s[sdj]))

        # 2) Isolation Forest (normal'de eğit, anomaly score)
        iso=IsolationForest(n_estimators=200, random_state=seed, n_jobs=1).fit(Xb)
        s_iso = -iso.score_samples(Ze)     # yüksek = anormal
        out["Isolation Forest"].append(roc_auc_score(yb, s_iso[sdj]))

        # 3) One-Class SVM (normal'de eğit, RBF)
        oc=OneClassSVM(kernel="rbf", gamma="scale", nu=0.1).fit(Xb)
        s_oc = -oc.score_samples(Ze)       # yüksek = anormal
        out["One-Class SVM"].append(roc_auc_score(yb, s_oc[sdj]))
    print("=== Lightweight baselines (source-rate, benign-trained, source-disjoint ROC) ===")
    for k,v in out.items():
        print(f"  {k:22s} {np.mean(v):.3f} ± {np.std(v):.3f}")
    print("\n>>> Hepsi ~yüksekse: source-rate temsili + benign-anchor belirleyici,")
    print(">>> spesifik dedektör değil (R3.8 + R2.4). Attention'ın rolü kararlılık (§5.2).")
    return out

run_lightweight()

=== Lightweight baselines (source-rate, benign-trained, source-disjoint ROC) ===
  benign-Mahalanobis     0.907 ± 0.002
  Isolation Forest       0.728 ± 0.010
  One-Class SVM          0.971 ± 0.001

>>> Hepsi ~yüksekse: source-rate temsili + benign-anchor belirleyici,
>>> spesifik dedektör değil (R3.8 + R2.4). Attention'ın rolü kararlılık (§5.2).


{'benign-Mahalanobis': [np.float64(0.9052932189749289),
  np.float64(0.9054166614200893),
  np.float64(0.9048476714338611),
  np.float64(0.9105501789370423),
  np.float64(0.9088436755216953)],
 'Isolation Forest': [np.float64(0.7367273571375532),
  np.float64(0.7124846742367461),
  np.float64(0.7255679776949993),
  np.float64(0.7419931001607832),
  np.float64(0.7245496169475059)],
 'One-Class SVM': [np.float64(0.9699464895386521),
  np.float64(0.9717045971239516),
  np.float64(0.9712791690834877),
  np.float64(0.9710642717241886),
  np.float64(0.970620951679064)]}